<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_04_1_kfold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab öffnen"/></a>


# T81-558: Anwendungen tiefer neuronaler Netzwerke

**Modul 4: Training für tabellarische Daten**

- Dozent: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Jeff Heaton](https://sites.wustl.edu/jeffheaton/)
– Weitere Informationen finden Sie unter [class website](https://sites.wustl.edu/jeffheaton/t81-558/).


# Modul 4 Material

- **Teil 4.1: Verwenden der K-fachen Kreuzvalidierung mit PyTorch** [[Video]](https://www.youtube.com/watch?v=Q8ZQNvZwsNE&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=Q8ZQNvZwsNE&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
- Teil 4.2: Trainingspläne für PyTorch [[Video]](https://www.youtube.com/watch?v=lMMlbmfvKDQ&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=lMMlbmfvKDQ&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
- Teil 4.3: Dropout-Regularisierung [[Video]](https://www.youtube.com/watch?v=4ixjgw6Q42U&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=4ixjgw6Q42U&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
- Teil 4.4: Batch-Normalisierung [[Video]](https://www.youtube.com/watch?v=1U5nOKh9OLQ&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=1U5nOKh9OLQ&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
- Teil 4.5: RAPIDS für tabellarische Daten [[Video]](https://www.youtube.com/watch?v=KgoXuhG_kfs&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=KgoXuhG_kfs&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)


# Google CoLab-Anweisungen

Der folgende Code stellt sicher, dass Google CoLab ausgeführt wird und ordnet bei Bedarf Google Drive zu. Wir initialisieren das PyTorch-Gerät auch entweder auf GPU/MPS (falls verfügbar) oder CPU.


In [1]:
import copy
import torch

try:
    import google.colab

    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Nutzen Sie eine GPU oder MPS (Apple), sofern verfügbar. (siehe Modul 3.2)
import torch

has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Verwendetes Gerät: {device}")


# Frühzeitiges Stoppen (siehe Modul 3.4)
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_model = None
        self.best_loss = None
        self.counter = 0
        self.status = ""

    def __call__(self, model, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model = copy.deepcopy(model.state_dict())
        elif self.best_loss - val_loss >= self.min_delta:
            self.best_model = copy.deepcopy(model.state_dict())
            self.best_loss = val_loss
            self.counter = 0
            self.status = f"Improvement found, counter reset to {self.counter}"
        else:
            self.counter += 1
            self.status = f"No improvement in the last {self.counter} epochs"
            if self.counter >= self.patience:
                self.status = f"Early stopping triggered after {self.counter} epochs."
                if self.restore_best_weights:
                    model.load_state_dict(self.best_model)
                return True
        return False

Note: not using Google CoLab
Using device: mps


# Teil 4.1: Verwenden der K-fachen Kreuzvalidierung mit PyTorch

Sie können die Kreuzvalidierung bei der prädiktiven Modellierung für verschiedene Zwecke verwenden:

- Generieren von Out-of-Sample-Vorhersagen aus einem neuronalen Netzwerk
- Schätzen Sie eine gute Anzahl von Epochen, für die ein neuronales Netzwerk trainiert werden soll (frühzeitiges Stoppen).
- Bewerten Sie die Wirksamkeit bestimmter Hyperparameter wie Aktivierungsfunktionen, Neuronenzahlen und Schichtzahlen

Bei der Kreuzvalidierung werden mehrere Faltungen und mehrere Modelle verwendet, um jedem Datensegment die Möglichkeit zu geben, sowohl als Validierungs- als auch als Trainingssatz zu dienen. Abbildung 5.CROSS zeigt die Kreuzvalidierung.

**Abbildung 5.CROSS: K-fache Kreuzvalidierung**
![K-Fold Crossvalidation](https://raw.githubusercontent.com/jeffheaton/t81_558_deep_learning/master/images/class_1_kfold.png "K-Fold Crossvalidation")

Es ist wichtig zu beachten, dass jede Falte ein Modell (neuronales Netzwerk) hat. Um Vorhersagen für neue Daten (die nicht im Trainingssatz vorhanden sind) zu generieren, können Vorhersagen aus den Faltmodellen auf verschiedene Arten verarbeitet werden:

- Wählen Sie das Modell mit der höchsten Validierungspunktzahl als endgültiges Modell aus.
- Legen Sie für die fünf Modelle neue Daten vorab fest (eines für jede Falte) und berechnen Sie den Durchschnitt der Ergebnisse (dies ist ein [ensemble](https://en.wikipedia.org/wiki/Ensemble_learning)).
- Trainieren Sie ein neues Modell (mit denselben Einstellungen wie bei der Kreuzvalidierung) anhand des gesamten Datensatzes. Trainieren Sie für so viele Epochen wie möglich und mit derselben Struktur der verborgenen Schicht.

Im Allgemeinen bevorzuge ich den letzten Ansatz und trainiere ein Modell erneut mit dem gesamten Datensatz, nachdem ich die Hyperparameter ausgewählt habe. Natürlich werde ich immer einen letzten Holdout-Satz für die Modellvalidierung beiseite legen, den ich in keinem Aspekt des Trainingsprozesses verwende.

## Regression vs. Klassifizierung K-fach Kreuzvalidierung

Regression und Klassifizierung werden bei der Kreuzvalidierung etwas anders behandelt. Regression ist der einfachere Fall, bei dem Sie den Datensatz in K-Falten aufteilen können, ohne sich groß darum zu kümmern, wo jedes Element landet. Bei der Regression sollten die Datenelemente so zufällig wie möglich in die Falten fallen. Es ist auch wichtig zu bedenken, dass nicht jede Falte unbedingt die gleiche Anzahl von Datenelementen enthält. Es ist nicht immer möglich, den Datensatz gleichmäßig in K-Falten aufzuteilen. Für die Regressionskreuzvalidierung verwenden wir die Scikit-Learn-Klasse **KFold**.

Bei der Kreuzvalidierung zur Klassifizierung könnte auch das **KFold**-Objekt verwendet werden. Diese Technik würde jedoch nicht sicherstellen, dass die Klassenbalance in jeder Falte dieselbe bleibt wie im Original. Die Balance der Klassen, mit denen ein Modell trainiert wurde, muss dieselbe bleiben wie im Trainingsset (oder ähnlich). Die Abweichung in dieser Verteilung ist eines der wichtigsten Dinge, die überwacht werden müssen, nachdem ein trainiertes Modell tatsächlich verwendet wurde. Aus diesem Grund möchten wir sicherstellen, dass die Kreuzvalidierung selbst keine unbeabsichtigte Verschiebung einführt. Diese Technik wird als geschichtete Stichprobennahme bezeichnet und wird erreicht, indem bei jeder Klassifizierung das Scikit-Learn-Objekt **StratifiedKFold** anstelle von **KFold** verwendet wird. Zusammenfassend sollten Sie in Scikit-Learn die folgenden beiden Objekte verwenden:

- **KFold** Beim Umgang mit einem Regressionsproblem.
- **StratifiedKFold** Bei der Behandlung eines Klassifizierungsproblems.

Die folgenden beiden Abschnitte demonstrieren die Kreuzvalidierung mit Klassifizierung und Regression.

## Out-of-Sample-Regressionsvorhersagen mit K-facher Kreuzvalidierung

Der folgende Code trainiert den einfachen Datensatz mithilfe einer 5-fachen Kreuzvalidierung. Die erwartete Leistung eines neuronalen Netzwerks des hier trainierten Typs wäre die Punktzahl für die generierten Out-of-Sample-Vorhersagen. Wir beginnen mit der Vorbereitung eines Merkmalsvektors mithilfe des **jh-simple-dataset** zur Vorhersage des Alters. Dieses Modell ist als Regressionsproblem angelegt.


In [2]:
import pandas as pd
from scipy.stats import zscore
from sklearn.model_selection import train_test_split

# Lesen des Datensatzes
df = read_csv(
    "https://data.heatonresearch.com/data/t81-558/jh-simple-dataset.csv",
    na_values=["NA", "?"],
)

# Dummies für den Job generieren
df = concat([df, get_dummies(df["job"], prefix="job", dtype=int)], axis=1)
df.drop("job", axis=1, inplace=True)

# Dummies für den Bereich generieren
df = concat([df, get_dummies(df["area"], prefix="area", dtype=int)], axis=1)
df.drop("area", axis=1, inplace=True)

# Dummies für Produkte generieren
df = concat([df, get_dummies(df["product"], prefix="product", dtype=int)], axis=1)
df.drop("product", axis=1, inplace=True)

# Fehlende Werte für Einkommen
med = df["income"].median()
df["income"] = df["income"].fillna(med)

# Bereiche standardisieren
df["income"] = zscore(df["income"])
df["aspect"] = zscore(df["aspect"])
df["save_rate"] = zscore(df["save_rate"])
df["subscriptions"] = zscore(df["subscriptions"])

Nachdem der Merkmalsvektor erstellt wurde, kann eine 5-fache Kreuzvalidierung durchgeführt werden, um Out-of-Sample-Vorhersagen zu generieren. Wir gehen von 500 Epochen aus und verwenden kein frühzeitiges Stoppen. Später werden wir sehen, wie wir eine optimalere Epochenanzahl schätzen können.


In [3]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

# In PyTorch-Tensoren konvertieren
x_columns = df.columns.drop(["age", "id"])
x = torch.tensor(df[x_columns].values, dtype=torch.float32, device=device)
y = torch.tensor(df["age"].values, dtype=torch.float32, device=device).view(-1, 1)

# Legen Sie einen Zufallsstartwert für die Reproduzierbarkeit fest
torch.manual_seed(42)

# Kreuzvalidierung
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Parameter für vorzeitiges Stoppen
patience = 10

fold = 0
for train_idx, test_idx in kf.split(x):
    fold += 1
    print(f"Fold # {falten}")

    x_train, x_test = x[train_idx], x[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # PyTorch DataLoader
    train_dataset = TensorDataset(x_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    # Erstellen des Modells und des Optimierers
    model = nn.Sequential(
        nn.Linear(x.shape[1], 20),
        nn.ReLU(),
        nn.Linear(20, 10),
        nn.ReLU(),
        nn.Linear(10, 1),
    )
    model = torch.compile(model, backend="aot_eager").to(device)

    optimizer = optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()

    # Variablen für frühzeitiges Stoppen
    best_loss = float("inf")
    early_stopping_counter = 0

    # Trainingsschleife
    EPOCHS = 500
    epoch = 0
    done = False
    es = EarlyStopping()

    while not done and epoch < EPOCHS:
        epoch += 1
        model.train()
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad()
            output = model(x_batch)
            loss = loss_fn(output, y_batch)
            loss.backward()
            optimizer.step()

        # Validierung
        model.eval()
        with torch.no_grad():
            val_output = model(x_test)
            val_loss = loss_fn(val_output, y_test)

        if es(model, val_loss):
            done = True

    print(
        f"Epoch {epoch}/{EPOCHS}, Validation Loss: " f"{val_loss.item()}, {es.status}"
    )

# Abschließende Bewertung
model.eval()
with torch.no_grad():
    oos_pred = model(x_test)
score = torch.sqrt(loss_fn(oos_pred, y_test)).item()
print(f"Faltungsbewertung (RMSE): {score}")

Fold #1


/Users/jeff/miniconda3/envs/torch/lib/python3.9/site-packages/torch/autograd/__init__.py:394: UserWarning: Error detected in LinearBackward0. Traceback of forward call that caused the error:
  File "/Users/jeff/miniconda3/envs/torch/lib/python3.9/site-packages/torch/nn/modules/container.py", line 215, in forward
    input = module(input)
 (Triggered internally at /Users/runner/work/_temp/anaconda/conda-bld/pytorch_1702400227158/work/torch/csrc/autograd/python_anomaly_mode.cpp:119.)
  result = Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
[2024-01-07 07:01:12,347] [0/2] torch._dynamo.exc: [WARNING] Backend compiler failed with a fake tensor exception at 
[2024-01-07 07:01:12,347] [0/2] torch._dynamo.exc: [WARNING]   File "/Users/jeff/miniconda3/envs/torch/lib/python3.9/site-packages/torch/nn/modules/container.py", line 216, in forward
[2024-01-07 07:01:12,347] [0/2] torch._dynamo.exc: [WARNING]     return input
[2024-01-07 07:01:12,347] [

Epoch 157/500, Validation Loss: 0.7110840678215027, Early stopping triggered after 5 epochs.
Fold #2
Epoch 149/500, Validation Loss: 0.4980887770652771, Early stopping triggered after 5 epochs.
Fold #3
Epoch 151/500, Validation Loss: 0.7315194010734558, Early stopping triggered after 5 epochs.
Fold #4
Epoch 191/500, Validation Loss: 0.42675867676734924, Early stopping triggered after 5 epochs.
Fold #5
Epoch 139/500, Validation Loss: 1.247514009475708, Early stopping triggered after 5 epochs.
Fold score (RMSE): 1.1104768514633179


Wie Sie sehen, gibt der obige Code auch die durchschnittliche Anzahl der benötigten Epochen an. Eine gängige Technik besteht darin, dann mit dem gesamten Datensatz für die durchschnittliche Anzahl der benötigten Epochen zu trainieren.

## Klassifizierung mit geschichteter K-facher Kreuzvalidierung

Der folgende Code trainiert und passt den **jh**-simple-dataset-Datensatz mit Kreuzvalidierung an, um Out-of-Sample-Ergebnisse zu generieren. Er schreibt auch die Out-of-Sample-Ergebnisse (Vorhersagen für den Testsatz).

Es empfiehlt sich, eine stratifizierte K-fach-Kreuzvalidierung mit Klassifizierungsdaten durchzuführen. Diese Technik stellt sicher, dass die Prozentsätze jeder Klasse über alle Faltungen hinweg gleich bleiben. Verwenden Sie das **StratifiedKFold**-Objekt anstelle des **KFold**-Objekts, das in der Regression verwendet wird.


In [4]:
# Lesen des Datensatzes
df = read_csv(
    "https://data.heatonresearch.com/data/t81-558/jh-simple-dataset.csv",
    na_values=["NA", "?"],
)

# Dummies für den Job generieren
df = concat([df, get_dummies(df["job"], prefix="job", dtype=int)], axis=1)
df.drop("job", axis=1, inplace=True)

# Dummies für den Bereich generieren
df = concat([df, get_dummies(df["area"], prefix="area", dtype=int)], axis=1)
df.drop("area", axis=1, inplace=True)

# Fehlende Werte für Einkommen
med = df["income"].median()
df["income"] = df["income"].fillna(med)

# Bereiche standardisieren
df["income"] = zscore(df["income"])
df["aspect"] = zscore(df["aspect"])
df["save_rate"] = zscore(df["save_rate"])
df["age"] = zscore(df["age"])
df["subscriptions"] = zscore(df["subscriptions"])

# In Numpy konvertieren - Klassifizierung
x_columns = df.columns.drop("product").drop("id")
x = df[x_columns].values
dummies = get_dummies(df["product"], dtype=int)  # Einstufung
products = dummies.columns
y = dummies.values

Wir durchlaufen nun die fünf Faltungen und verwenden die Validierungsdaten in jeder Faltung zum frühzeitigen Stoppen. Wir behalten auch die validierten Vorhersagen bei, um einen vollständigen Satz von „Out-of-Sample“-Vorhersagen zu erstellen. Diese „Out-of-Sample“-Vorhersagen ermöglichen es uns, Vorhersagen über den gesamten Datensatz hinweg zu haben, die nicht in den Trainingsdaten enthalten waren. Es ist wichtig zu beachten, dass diese Trennung nicht 100 % rein ist, da der Validierungssatz zum frühzeitigen Stoppen verwendet wurde. Diese kleine Überkreuzung ist ein Kompromiss, der es uns ermöglicht, in jeder Faltung eine große Menge an Trainingsdaten zu verwenden.

In [5]:
import numpy as np
from sklearn import metrics
from sklearn.model_selection import StratifiedKFold

# Vorausgesetzt, Ihre Daten liegen in Numpy-Arrays vor. Wenn nicht, konvertieren Sie sie in Numpy-Arrays
x = np.array(x)
y = np.array(y)

# Verwenden Sie die nn.Sequential-API
model = nn.Sequential(
    nn.Linear(x.shape[1], 50),
    nn.ReLU(),
    nn.Linear(50, 25),
    nn.ReLU(),
    nn.Linear(25, y.shape[1]),
    nn.Softmax(dim=1),
)
model = torch.compile(model, backend="aot_eager").to(device)

# Kreuzvalidierung
kf = StratifiedKFold(5, shuffle=True, random_state=42)

# Definition von Verlust und Optimierer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

oos_y = []
oos_pred = []
fold = 0

for train, test in kf.split(x, df["product"]):
    fold += 1
    print(f"Fold # {falten}")

    x_train = torch.tensor(x[train], device=device, dtype=torch.float32)
    y_train = torch.tensor(
        np.argmax(y[train], axis=1), device=device, dtype=torch.long
    )  # In Klassenindizes konvertieren
    x_test = torch.tensor(x[test], device=device, dtype=torch.float32)
    y_test = torch.tensor(
        np.argmax(y[test], axis=1), device=device, dtype=torch.long
    )  # In Klassenindizes konvertieren

    # Trainingsschleife
    EPOCHS = 500
    epoch = 0
    done = False
    es = EarlyStopping(restore_best_weights=True)

    while not done and epoch < EPOCHS:
        epoch += 1
        model.train()
        optimizer.zero_grad()
        output = model(x_train)
        loss = criterion(output, y_train)
        loss.backward()
        optimizer.step()

        # Validierungsverlust auswerten
        model.eval()
        with torch.no_grad():
            y_val = model(x_test)
            val_loss = criterion(y_val, y_test)

        if es(model, val_loss):
            done = True

    # Vorhersage
    with torch.no_grad():
        y_val = model(x_test)
        _, pred = torch.max(y_val, 1)

    oos_y.append(y_test.cpu().numpy())
    oos_pred.append(pred.cpu().numpy())

    print(
        f"Epoch {epoch}/{EPOCHS}, Validation Loss: " f"{val_loss.item()}, {es.status}"
    )

    # Messen Sie die Genauigkeit dieser Falte
    score = metrics.accuracy_score(y_test.cpu().numpy(), pred.cpu().numpy())
print(f"Faltpunktzahl (Genauigkeit): {score}")

# Erstellen Sie die OOS-Vorhersageliste und berechnen Sie den Fehler.
oos_y = np.concatenate(oos_y)
oos_pred = np.concatenate(oos_pred)

score = metrics.accuracy_score(oos_y, oos_pred)
print(f"Endergebnis (Genauigkeit): {score}")

Fold #1
Epoch 395/500, Validation Loss: 1.4793496131896973, Early stopping triggered after 5 epochs.
Fold score (accuracy): 0.69
Fold #2
Epoch 6/500, Validation Loss: 1.4524176120758057, Early stopping triggered after 5 epochs.
Fold score (accuracy): 0.73
Fold #3
Epoch 6/500, Validation Loss: 1.4165765047073364, Early stopping triggered after 5 epochs.
Fold score (accuracy): 0.765
Fold #4
Epoch 6/500, Validation Loss: 1.4107660055160522, Early stopping triggered after 5 epochs.
Fold score (accuracy): 0.7625
Fold #5
Epoch 7/500, Validation Loss: 1.4483850002288818, Early stopping triggered after 5 epochs.
Fold score (accuracy): 0.7275
Final score (accuracy): 0.735


# Modul 4 Aufgabe

Die dritte Aufgabe finden Sie hier: [assignment 4](https://github.com/jeffheaton/app_deep_learning/blob/main/assignments/assignment_yourname_t81_558_class4.ipynb)